## cấu trúc
```
PromotionSyncService
        ↓
ISyncStrategy
   ↓        ↓
RK7DirectStrategy
RefAgentStrategy
```

## flow
```
promotion
   ↓
detect rule variant
   ↓
resolve strategy
   ↓
strategy.apply()
   ↓
final discount
```

In [5]:
// rk7_sync_strategy_example.cs

using System;
using System.Collections.Generic;
using System.Threading.Tasks;


// =========================================================
// DOMAIN MODEL
// =========================================================

public class PromotionSyncContext
{
    public string PromotionName { get; set; }

    public string RK7Payload { get; set; }

    public string Route { get; set; }
}


// =========================================================
// STRATEGY CONTRACT
// =========================================================

public interface ISyncStrategy
{
    Task Execute(PromotionSyncContext context);
}


// =========================================================
// DIRECT RK7 STRATEGY
// =========================================================

public class RK7DirectStrategy : ISyncStrategy
{
    public async Task Execute(PromotionSyncContext context)
    {
        Console.WriteLine("sync direct → rk7");

        await CallRK7Api(context.RK7Payload);
    }

    private Task CallRK7Api(string payload)
    {
        Console.WriteLine("calling rk7 api...");
        Console.WriteLine(payload);

        return Task.CompletedTask;
    }
}


// =========================================================
// REFAGENT STRATEGY
// =========================================================

public class RefAgentStrategy : ISyncStrategy
{
    public async Task Execute(PromotionSyncContext context)
    {
        Console.WriteLine("sync via refagent");

        await CallRefAgent(context.RK7Payload);
    }

    private async Task CallRefAgent(string payload)
    {
        Console.WriteLine("calling refagent api...");

        // refagent sau đó gọi rk7
        await CallRK7FromRefAgent(payload);
    }

    private Task CallRK7FromRefAgent(string payload)
    {
        Console.WriteLine("refagent → rk7");

        return Task.CompletedTask;
    }
}


// =========================================================
// STRATEGY RESOLVER
// =========================================================

public static class SyncStrategyResolver
{
    private static readonly Dictionary<string, ISyncStrategy> strategies =
        new Dictionary<string, ISyncStrategy>()
    {
        { "RK7_DIRECT", new RK7DirectStrategy() },
        { "REFAGENT", new RefAgentStrategy() }
    };

    public static ISyncStrategy Resolve(string route)
    {
        if (!strategies.ContainsKey(route))
            throw new Exception("route not supported");

        return strategies[route];
    }
}


// =========================================================
// SYNC SERVICE
// =========================================================

public class PromotionSyncService
{
    public async Task Sync(PromotionSyncContext context)
    {
        var strategy =
            SyncStrategyResolver.Resolve(context.Route);

        await strategy.Execute(context);
    }
}


// =========================================================
// DEMO
// =========================================================


var context = new PromotionSyncContext
{
    PromotionName = "summer promo",
    RK7Payload = "<promotion>...</promotion>",
    Route = "RK7_DIRECT"
};


var service = new PromotionSyncService();

await service.Sync(context);

sync direct → rk7
calling rk7 api...
<promotion>...</promotion>
